# Live-to-Canonical Station Mapping – Objective & Plan

## Objective
Retrieve live BIXI station information from `info_url` and map every live station to the cleaned canonical station mapping built in `01_dataeng_stations.ipynb`.

## Plan (task-gated)
1. **Task 1:** Retrieve and flatten live station feed into a DataFrame.
2. **Task 2:** Load canonical mapping artifact exported from the cleaning notebook.
3. **Task 3:** Match live stations to canonical stations using normalized name + coordinate proximity rules.
4. **Task 4:** Report match coverage, unmatched stations, and likely conflict cases.

> Following your instruction: each task will be executed and checked before moving to the next.

In [1]:
import requests
import re

In [2]:
# URLs
info_url   = "https://gbfs.velobixi.com/gbfs/2-2/en/station_information.json"

# Load JSON
info_data   = requests.get(info_url).json()
info_data

{'last_updated': 1773888367,
 'ttl': 10,
 'version': '2.2',
 'data': {'stations': [{'station_id': '3',
    'external_id': '0b0fe114-08f3-11e7-a1cb-3863bb33a4e4',
    'name': 'Clark / Ontario',
    'short_name': '6003',
    'lat': 45.510493862520995,
    'lon': -73.56692067715812,
    'rental_methods': ['CREDITCARD', 'KEY'],
    'capacity': 19,
    'electric_bike_surcharge_waiver': False,
    'is_charging': False,
    'eightd_has_key_dispenser': False,
    'has_kiosk': True},
   {'station_id': '15',
    'external_id': '0b0ffa68-08f3-11e7-a1cb-3863bb33a4e4',
    'name': 'Métro Berri-UQAM (St-Denis / de Maisonneuve)',
    'short_name': '6015',
    'lat': 45.51425199763688,
    'lon': -73.56150217976847,
    'rental_methods': ['CREDITCARD', 'KEY'],
    'capacity': 15,
    'electric_bike_surcharge_waiver': False,
    'is_charging': False,
    'eightd_has_key_dispenser': False,
    'has_kiosk': True},
   {'station_id': '17',
    'external_id': '0b0ffd3a-08f3-11e7-a1cb-3863bb33a4e4',
    'nam

In [3]:
# Task 1: flatten live station feed
import pandas as pd

live_stations_df = pd.DataFrame(info_data['data']['stations']).copy()

required_live_cols = ['station_id', 'name', 'capacity', 'lat', 'lon']
missing_live_cols = [c for c in required_live_cols if c not in live_stations_df.columns]

print(f"Live stations retrieved: {len(live_stations_df):,}")
print(f"Columns available: {len(live_stations_df.columns)}")
if missing_live_cols:
    print(f"Missing required live columns: {missing_live_cols}")
else:
    print("Required live columns present: station_id, name, capacity, lat, lon")

live_stations_df[['station_id', 'name', 'capacity', 'lat', 'lon']].head(10)

Live stations retrieved: 307
Columns available: 12
Required live columns present: station_id, name, capacity, lat, lon


,station_id,name,capacity,lat,lon
0,3,Clark / Ontario,19,45.510494,-73.566921
1,15,Métro Berri-UQAM (St-Denis / de Maisonneuve),15,45.514252,-73.561502
2,17,Marché St-Jacques (Atateken),27,45.520666,-73.563915
3,19,Métro Sherbrooke (de Rigaud / Berri),27,45.518093,-73.568026
4,22,de la Commune / des Soeurs-Grises,67,45.497929,-73.552732
5,24,Notre-Dame / St-Gabriel,11,45.507173,-73.555087
6,25,de la Commune / Place Jacques-Cartier,81,45.507610,-73.551836
7,26,de Maisonneuve / Mansfield (sud),27,45.501827,-73.573642
8,31,Métro Place-d'Armes (St-Urbain / Viger),19,45.506323,-73.559699
9,34,Viger / Chenneville,37,45.505382,-73.560937


In [4]:
# Task 2: load canonical mapping artifact from Spark station_cleaning outputs
from pathlib import Path

mapping_parquet_dir = Path("data/silver/station_cleaning/station_direct_match_mapping")
mapping_csv_dir = Path("data/silver/station_cleaning/station_direct_match_mapping_csv")

if mapping_parquet_dir.exists():
    canonical_mapping_df = pd.read_parquet(mapping_parquet_dir)
    mapping_source = str(mapping_parquet_dir)
elif mapping_csv_dir.exists():
    csv_parts = sorted(mapping_csv_dir.glob("part-*.csv"))
    if not csv_parts:
        raise FileNotFoundError(f"No part CSV files found in: {mapping_csv_dir}")
    canonical_mapping_df = pd.read_csv(csv_parts[0])
    mapping_source = str(csv_parts[0])
else:
    raise FileNotFoundError(
        "Missing mapping artifact. Expected one of: "
        f"{mapping_parquet_dir} or {mapping_csv_dir}"
    )

required_map_cols = [
    'canonical_station_id',
    'canonical_lat',
    'canonical_lon',
    'normalized_name',
    'coord_key',
]
missing_map_cols = [c for c in required_map_cols if c not in canonical_mapping_df.columns]

print(f"Mapping source: {mapping_source}")
print(f"Canonical mapping rows: {len(canonical_mapping_df):,}")
print(f"Canonical stations: {canonical_mapping_df['canonical_station_id'].nunique():,}")
if missing_map_cols:
    print(f"Missing required mapping columns: {missing_map_cols}")
else:
    print("Required mapping columns present")

canonical_mapping_df[required_map_cols].head(10)

Mapping source: data/silver/station_cleaning/station_direct_match_mapping
Canonical mapping rows: 1,901
Canonical stations: 1,423
Required mapping columns present


,canonical_station_id,canonical_lat,canonical_lon,normalized_name,coord_key
0,STN_1371,0.000000,0.000000,9599 (test cyclochrome),"0.000000,0.000000"
1,STN_1371,0.000000,0.000000,cyclo pbsc,"0.000000,0.000000"
2,STN_1371,0.000000,0.000000,cyclo winter station,"0.000000,0.000000"
3,STN_1371,0.000000,0.000000,mtl-eco4-lab-03,"0.000000,0.000000"
4,STN_1371,0.000000,0.000000,mtl-lab-20-docks,"0.000000,0.000000"
5,STN_1371,0.000000,0.000000,test eco5.1.1,"0.000000,0.000000"
6,STN_1203,45.379756,-71.928778,u. de sherbrooke (pavillon de la vie étudiante...,"45.379756,-71.928778"
7,STN_1203,45.379756,-71.928778,u. de sherbrooke (registrariat-voie 3),"45.379756,-71.928778"
8,STN_1196,45.379850,-71.924395,u. de sherbrooke (pavillon marie-victorin-voie 1),"45.379850,-71.924395"
9,STN_1196,45.379850,-71.924395,u. de sherbrooke (école de musique-voie 1),"45.379850,-71.924395"


In [5]:
# Task 3 + Task 4: map live stations to canonical stations and report coverage
import numpy as np


def normalize_station_name(text: str) -> str:
    if text is None:
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*/\s*", " / ", text)
    text = re.sub(r"\s*-\s*", "-", text)
    return text.strip()


def haversine_km_vec(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


MAX_NEAREST_KM = 0.05
MAX_NAME_COORD_KM = 0.20

live_map_df = live_stations_df.copy()
live_map_df['normalized_name'] = live_map_df['name'].map(normalize_station_name)
live_map_df['coord_key'] = live_map_df['lat'].map(lambda v: f"{float(v):.6f}") + "," + live_map_df['lon'].map(lambda v: f"{float(v):.6f}")

canon_unique_df = (
    canonical_mapping_df[['canonical_station_id', 'canonical_lat', 'canonical_lon', 'normalized_name', 'coord_key']]
    .drop_duplicates()
    .copy()
)

# Stage A: exact coordinate key
coord_lookup_df = canon_unique_df[['coord_key', 'canonical_station_id']].drop_duplicates('coord_key')
live_mapped_df = live_map_df.merge(coord_lookup_df, how='left', on='coord_key')
live_mapped_df['match_method'] = np.where(live_mapped_df['canonical_station_id'].notna(), 'exact_coord_key', None)
live_mapped_df['match_distance_km'] = np.where(live_mapped_df['canonical_station_id'].notna(), 0.0, np.nan)

# Stage B: normalized name exact + nearest coord within threshold
unmatched_idx = live_mapped_df[live_mapped_df['canonical_station_id'].isna()].index.tolist()
if unmatched_idx:
    by_name = {k: v for k, v in canon_unique_df.groupby('normalized_name')}
    for idx in unmatched_idx:
        row = live_mapped_df.loc[idx]
        name = row['normalized_name']
        if name not in by_name:
            continue
        cand = by_name[name]
        dists = haversine_km_vec(
            row['lat'], row['lon'],
            cand['canonical_lat'].to_numpy(), cand['canonical_lon'].to_numpy()
        )
        min_pos = int(np.argmin(dists))
        min_dist = float(dists[min_pos])
        if min_dist <= MAX_NAME_COORD_KM:
            live_mapped_df.at[idx, 'canonical_station_id'] = cand.iloc[min_pos]['canonical_station_id']
            live_mapped_df.at[idx, 'match_method'] = 'exact_normalized_name_nearest_coord'
            live_mapped_df.at[idx, 'match_distance_km'] = min_dist

# Stage C: nearest canonical station within MAX_NEAREST_KM
unmatched_idx = live_mapped_df[live_mapped_df['canonical_station_id'].isna()].index.tolist()
if unmatched_idx:
    canon_station_centers = (
        canon_unique_df[['canonical_station_id', 'canonical_lat', 'canonical_lon']]
        .drop_duplicates('canonical_station_id')
        .reset_index(drop=True)
    )
    center_lats = canon_station_centers['canonical_lat'].to_numpy()
    center_lons = canon_station_centers['canonical_lon'].to_numpy()

    for idx in unmatched_idx:
        row = live_mapped_df.loc[idx]
        dists = haversine_km_vec(row['lat'], row['lon'], center_lats, center_lons)
        min_pos = int(np.argmin(dists))
        min_dist = float(dists[min_pos])
        if min_dist <= MAX_NEAREST_KM:
            live_mapped_df.at[idx, 'canonical_station_id'] = canon_station_centers.iloc[min_pos]['canonical_station_id']
            live_mapped_df.at[idx, 'match_method'] = 'nearest_canonical_within_0.05km'
            live_mapped_df.at[idx, 'match_distance_km'] = min_dist

# Coverage report
mapped_count = int(live_mapped_df['canonical_station_id'].notna().sum())
total_live = int(len(live_mapped_df))
unmapped_count = total_live - mapped_count
coverage_pct = 100.0 * mapped_count / total_live if total_live else 0.0

match_method_counts_df = (
    live_mapped_df
    .assign(match_method=live_mapped_df['match_method'].fillna('unmatched'))
    .groupby('match_method', as_index=False)
    .agg(station_count=('station_id', 'count'))
    .sort_values('station_count', ascending=False)
)

unmatched_live_stations_df = (
    live_mapped_df[live_mapped_df['canonical_station_id'].isna()]
    [['station_id', 'name', 'normalized_name', 'lat', 'lon']]
    .sort_values('name')
)

print(f"Live stations total: {total_live}")
print(f"Mapped stations: {mapped_count}")
print(f"Unmapped stations: {unmapped_count}")
print(f"Coverage: {coverage_pct:.2f}%")

print("\nMatch method breakdown:")
display(match_method_counts_df)

if unmapped_count > 0:
    print("\nUnmapped live stations (first 30):")
    display(unmatched_live_stations_df.head(30))
else:
    print("All live stations mapped successfully.")

live_to_canonical_mapping_df = live_mapped_df[['station_id', 'name', 'normalized_name', 'lat', 'lon', 'canonical_station_id', 'match_method', 'match_distance_km']].copy()
display(live_to_canonical_mapping_df.head(30))

Live stations total: 307
Mapped stations: 307
Unmapped stations: 0
Coverage: 100.00%

Match method breakdown:


,match_method,station_count
0,exact_coord_key,298
1,exact_normalized_name_nearest_coord,8
2,nearest_canonical_within_0.05km,1


All live stations mapped successfully.


,station_id,name,normalized_name,lat,lon,canonical_station_id,match_method,match_distance_km
0,3,Clark / Ontario,clark / ontario,45.510494,-73.566921,STN_0346,exact_normalized_name_nearest_coord,0.010890
1,15,Métro Berri-UQAM (St-Denis / de Maisonneuve),métro berri-uqam (st-denis / de maisonneuve),45.514252,-73.561502,STN_0990,exact_coord_key,0.000000
2,17,Marché St-Jacques (Atateken),marché st-jacques (atateken),45.520666,-73.563915,STN_0062,exact_coord_key,0.000000
3,19,Métro Sherbrooke (de Rigaud / Berri),métro sherbrooke (de rigaud / berri),45.518093,-73.568026,STN_0020,exact_normalized_name_nearest_coord,0.005808
4,22,de la Commune / des Soeurs-Grises,de la commune / des soeurs-grises,45.497929,-73.552732,STN_0460,exact_coord_key,0.000000
5,24,Notre-Dame / St-Gabriel,notre-dame / st-gabriel,45.507173,-73.555087,STN_0481,exact_normalized_name_nearest_coord,0.004048
6,25,de la Commune / Place Jacques-Cartier,de la commune / place jacques-cartier,45.507610,-73.551836,STN_0006,exact_coord_key,0.000000
7,26,de Maisonneuve / Mansfield (sud),de maisonneuve / mansfield (sud),45.501827,-73.573642,STN_0125,exact_normalized_name_nearest_coord,0.028747
8,31,Métro Place-d'Armes (St-Urbain / Viger),métro place-d'armes (st-urbain / viger),45.506323,-73.559699,STN_0504,exact_coord_key,0.000000
9,34,Viger / Chenneville,viger / chenneville,45.505382,-73.560937,STN_0320,exact_normalized_name_nearest_coord,0.008555


### Task 5: Build a canonical station ID resolver
Objective: create a reusable resolver that returns canonical_station_id
When given raw station name, latitude, and longitude.

### Plan:
1) Define resolver class with same staged matching logic
2) Expose resolve(raw_name, lat, lon) request-style method
3) Run a sanity-check request and inspect output

In [6]:
# Task 5 implementation: request-style canonical station resolver
class CanonicalStationResolver:
    def __init__(self, canonical_mapping: pd.DataFrame, max_name_coord_km: float = 0.20, max_nearest_km: float = 0.05):
        required_cols = {
            'canonical_station_id',
            'canonical_lat',
            'canonical_lon',
            'normalized_name',
            'coord_key',
        }
        missing = required_cols - set(canonical_mapping.columns)
        if missing:
            raise ValueError(f"canonical_mapping is missing required columns: {sorted(missing)}")

        self.max_name_coord_km = float(max_name_coord_km)
        self.max_nearest_km = float(max_nearest_km)

        self.canon_unique_df = (
            canonical_mapping[['canonical_station_id', 'canonical_lat', 'canonical_lon', 'normalized_name', 'coord_key']]
            .dropna(subset=['canonical_station_id', 'canonical_lat', 'canonical_lon'])
            .drop_duplicates()
            .copy()
        )

        self.coord_lookup_df = (
            self.canon_unique_df[['coord_key', 'canonical_station_id']]
            .drop_duplicates('coord_key')
            .reset_index(drop=True)
        )
        self.coord_to_station = dict(zip(self.coord_lookup_df['coord_key'], self.coord_lookup_df['canonical_station_id']))
        self.by_name = {k: v.reset_index(drop=True) for k, v in self.canon_unique_df.groupby('normalized_name')}

        self.canon_station_centers = (
            self.canon_unique_df[['canonical_station_id', 'canonical_lat', 'canonical_lon']]
            .drop_duplicates('canonical_station_id')
            .reset_index(drop=True)
        )
        self.center_lats = self.canon_station_centers['canonical_lat'].to_numpy()
        self.center_lons = self.canon_station_centers['canonical_lon'].to_numpy()

    @staticmethod
    def _normalize_station_name(text: str) -> str:
        if text is None:
            return ""
        text = str(text).strip().lower()
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"\s*/\s*", " / ", text)
        text = re.sub(r"\s*-\s*", "-", text)
        return text.strip()

    @staticmethod
    def _haversine_km_vec(lat1, lon1, lat2, lon2):
        r = 6371.0
        lat1_rad = np.radians(lat1)
        lon1_rad = np.radians(lon1)
        lat2_rad = np.radians(lat2)
        lon2_rad = np.radians(lon2)
        dlat = lat2_rad - lat1_rad
        dlon = lon2_rad - lon1_rad
        a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
        return 2 * r * np.arcsin(np.sqrt(a))

    def resolve(self, raw_name: str, lat: float, lon: float) -> dict:
        normalized_name = self._normalize_station_name(raw_name)
        coord_key = f"{float(lat):.6f},{float(lon):.6f}"

        # Stage A: exact coord key
        station_id = self.coord_to_station.get(coord_key)
        if station_id is not None:
            return {
                'canonical_station_id': station_id,
                'match_method': 'exact_coord_key',
                'match_distance_km': 0.0,
                'normalized_name': normalized_name,
                'coord_key': coord_key,
            }

        # Stage B: normalized name + nearest coord
        if normalized_name in self.by_name:
            cand = self.by_name[normalized_name]
            dists = self._haversine_km_vec(
                float(lat), float(lon),
                cand['canonical_lat'].to_numpy(), cand['canonical_lon'].to_numpy(),
            )
            min_pos = int(np.argmin(dists))
            min_dist = float(dists[min_pos])
            if min_dist <= self.max_name_coord_km:
                return {
                    'canonical_station_id': cand.iloc[min_pos]['canonical_station_id'],
                    'match_method': 'exact_normalized_name_nearest_coord',
                    'match_distance_km': min_dist,
                    'normalized_name': normalized_name,
                    'coord_key': coord_key,
                }

        # Stage C: nearest canonical within threshold
        dists = self._haversine_km_vec(float(lat), float(lon), self.center_lats, self.center_lons)
        min_pos = int(np.argmin(dists))
        min_dist = float(dists[min_pos])
        if min_dist <= self.max_nearest_km:
            return {
                'canonical_station_id': self.canon_station_centers.iloc[min_pos]['canonical_station_id'],
                'match_method': 'nearest_canonical_within_0.05km',
                'match_distance_km': min_dist,
                'normalized_name': normalized_name,
                'coord_key': coord_key,
            }

        return {
            'canonical_station_id': None,
            'match_method': 'unmatched',
            'match_distance_km': np.nan,
            'normalized_name': normalized_name,
            'coord_key': coord_key,
        }


resolver = CanonicalStationResolver(canonical_mapping_df, max_name_coord_km=MAX_NAME_COORD_KM, max_nearest_km=MAX_NEAREST_KM)

sample = live_stations_df.iloc[0]
sample_result = resolver.resolve(sample['name'], sample['lat'], sample['lon'])
print('Sample request input:')
display(sample[['station_id', 'name', 'lat', 'lon']])
print('\nSample resolver output:')
sample_result

Sample request input:


station_id                  3
name          Clark / Ontario
lat                 45.510494
lon                -73.566921
Name: 0, dtype: object


Sample resolver output:


{'canonical_station_id': 'STN_0346',
 'match_method': 'exact_normalized_name_nearest_coord',
 'match_distance_km': 0.010889656511468624,
 'normalized_name': 'clark / ontario',
 'coord_key': '45.510494,-73.566921'}